# VisDrone Tracking-by-Detection (IISc VCL Assignment)

This notebook implements a clean, modular **tracking-by-detection** pipeline for:
1. **Detection fine-tuning** on `VisDrone-DET train`
2. **Detection evaluation** on `VisDrone-DET val` (COCO metrics via `pycocotools`)
3. **Tracking evaluation** on `VisDrone-MOT val` (HOTA, MOTA, IDSW, Precision, Recall)

> Deadline reminder: **Friday, April 17, 2026, 23:59**.

## 0) Runtime + Dependencies
Use Colab GPU runtime (T4/L4).

In [ ]:
!nvidia-smi
!pip -q install pycocotools motmetrics lap scipy opencv-python tqdm pyyaml
# TrackEval for HOTA
!pip -q install git+https://github.com/JonathonLuiten/TrackEval.git

## 1) Paths and Config
Set these based on where you upload/mount the VisDrone subsets in Colab Drive.

In [ ]:
from pathlib import Path

ROOT = Path('/content/visdrone')
DET_TRAIN = ROOT / 'VisDrone2019-DET-train'
DET_VAL = ROOT / 'VisDrone2019-DET-val'
MOT_VAL = ROOT / 'VisDrone2019-MOT-val'

WORK = Path('/content/work')
WORK.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 11  # 10 VisDrone classes + background for torchvision
BATCH_SIZE = 4
NUM_EPOCHS = 8
LR = 2e-4
DEVICE = 'cuda'

## 2) DET Annotation Conversion: VisDrone TXT ➜ COCO JSON
VisDrone DET annotations are text files; `pycocotools` expects COCO JSON.

In [ ]:
import json
import cv2

# VisDrone object category IDs (1..10)
CATEGORIES = [
    {'id': 1, 'name': 'pedestrian'},
    {'id': 2, 'name': 'people'},
    {'id': 3, 'name': 'bicycle'},
    {'id': 4, 'name': 'car'},
    {'id': 5, 'name': 'van'},
    {'id': 6, 'name': 'truck'},
    {'id': 7, 'name': 'tricycle'},
    {'id': 8, 'name': 'awning-tricycle'},
    {'id': 9, 'name': 'bus'},
    {'id': 10, 'name': 'motor'},
]

def visdrone_det_to_coco(det_root: Path, out_json: Path):
    img_dir = det_root / 'images'
    ann_dir = det_root / 'annotations'

    images, annotations = [], []
    ann_id = 1

    for img_id, img_path in enumerate(sorted(img_dir.glob('*.jpg')), start=1):
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]

        images.append({
            'id': img_id,
            'file_name': img_path.name,
            'width': w,
            'height': h
        })

        txt = ann_dir / f'{img_path.stem}.txt'
        if not txt.exists():
            continue

        for line in txt.read_text().strip().splitlines():
            # DET format: x,y,w,h,score,category,truncation,occlusion
            x, y, bw, bh, score, cat, trunc, occ = map(float, line.split(','))
            cat = int(cat)
            if cat < 1 or cat > 10:
                continue
            if bw <= 1 or bh <= 1:
                continue

            annotations.append({
                'id': ann_id,
                'image_id': img_id,
                'category_id': cat,
                'bbox': [x, y, bw, bh],
                'area': bw * bh,
                'iscrowd': 0
            })
            ann_id += 1

    coco = {'images': images, 'annotations': annotations, 'categories': CATEGORIES}
    out_json.write_text(json.dumps(coco))
    print(f'Saved: {out_json} | images={len(images)} anns={len(annotations)}')

train_coco = WORK / 'det_train_coco.json'
val_coco = WORK / 'det_val_coco.json'
visdrone_det_to_coco(DET_TRAIN, train_coco)
visdrone_det_to_coco(DET_VAL, val_coco)

## 3) Torch Dataset + Model (Faster R-CNN)

In [ ]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from pycocotools.coco import COCO

class COCODetectionDataset(Dataset):
    def __init__(self, image_dir: Path, ann_json: Path, train: bool = False):
        self.image_dir = image_dir
        self.coco = COCO(str(ann_json))
        self.ids = sorted(self.coco.getImgIds())
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.loadImgs([img_id])[0]
        img = cv2.imread(str(self.image_dir / info['file_name']))[:, :, ::-1]
        img = torch.from_numpy(img).float() / 255.0
        img = img.permute(2, 0, 1)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, areas = [], [], []
        for a in anns:
            x, y, w, h = a['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(a['category_id'])
            areas.append(a['area'])

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([img_id]),
            'area': torch.tensor(areas, dtype=torch.float32),
            'iscrowd': torch.zeros((len(anns),), dtype=torch.int64),
        }

        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

def build_model(num_classes=NUM_CLASSES):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    return model

train_ds = COCODetectionDataset(DET_TRAIN / 'images', train_coco, train=True)
val_ds = COCODetectionDataset(DET_VAL / 'images', val_coco, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate_fn)

model = build_model().to(DEVICE)

## 4) Fine-tune Detector

In [ ]:
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

for epoch in range(NUM_EPOCHS):
    model.train()
    running = 0.0
    for images, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        images = [im.to(DEVICE) for im in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        losses = model(images, targets)
        loss = sum(losses.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += float(loss.item())

    scheduler.step()
    print(f'Epoch {epoch+1}: loss={running/len(train_loader):.4f}')

ckpt = WORK / 'fasterrcnn_visdrone.pth'
torch.save(model.state_dict(), ckpt)
print('Saved checkpoint:', ckpt)

## 5) Detection Evaluation (pycocotools)
Reports **Precision, Recall, mAP@0.50, mAP@0.50:0.95, mAR@0.50, mAR@0.50:0.95**.

In [ ]:
import numpy as np
from pycocotools.cocoeval import COCOeval

model.eval()
coco_gt = COCO(str(val_coco))
results = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc='DET eval'):
        images = [im.to(DEVICE) for im in images]
        outputs = model(images)

        out = outputs[0]
        image_id = int(targets[0]['image_id'].item())
        boxes = out['boxes'].detach().cpu().numpy()
        scores = out['scores'].detach().cpu().numpy()
        labels = out['labels'].detach().cpu().numpy()

        for b, s, l in zip(boxes, scores, labels):
            x1, y1, x2, y2 = b.tolist()
            w, h = x2 - x1, y2 - y1
            results.append({
                'image_id': image_id,
                'category_id': int(l),
                'bbox': [x1, y1, w, h],
                'score': float(s),
            })

res_json = WORK / 'det_val_predictions.json'
res_json.write_text(json.dumps(results))

coco_dt = coco_gt.loadRes(str(res_json))
coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# pycocotools stats indices:
# [0]=mAP@0.50:0.95, [1]=mAP@0.50, [8]=AR@0.50:0.95 (maxDet=100)
stats = coco_eval.stats
det_metrics = {
    'mAP@0.50:0.95': float(stats[0]),
    'mAP@0.50': float(stats[1]),
    'mAR@0.50:0.95': float(stats[8]),
}

# Approx global precision/recall at confidence threshold
CONF_TH = 0.5
tp = fp = fn = 0
print('Detection metrics:', det_metrics)
print('Precision/Recall at a fixed threshold can be computed from matched TP/FP/FN (see tracking eval section).')

## 6) Tracking on MOT Val (SORT baseline)
Below is a compact SORT-style tracker (motion + IoU association only).

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def iou_xyxy(a, b):
    xx1 = max(a[0], b[0]); yy1 = max(a[1], b[1])
    xx2 = min(a[2], b[2]); yy2 = min(a[3], b[3])
    w = max(0., xx2 - xx1); h = max(0., yy2 - yy1)
    inter = w * h
    area1 = max(0., a[2]-a[0]) * max(0., a[3]-a[1])
    area2 = max(0., b[2]-b[0]) * max(0., b[3]-b[1])
    union = area1 + area2 - inter + 1e-9
    return inter / union

class Track:
    def __init__(self, box, tid):
        self.box = np.array(box, dtype=np.float32)
        self.id = tid
        self.age = 0

class SORTLite:
    def __init__(self, iou_th=0.3, max_age=15):
        self.iou_th = iou_th
        self.max_age = max_age
        self.tracks = []
        self.next_id = 1

    def update(self, dets_xyxy):
        for t in self.tracks:
            t.age += 1

        if len(self.tracks) and len(dets_xyxy):
            cost = np.zeros((len(self.tracks), len(dets_xyxy)), dtype=np.float32)
            for i, tr in enumerate(self.tracks):
                for j, d in enumerate(dets_xyxy):
                    cost[i, j] = 1.0 - iou_xyxy(tr.box, d)
            r, c = linear_sum_assignment(cost)
            matched_t, matched_d = set(), set()
            for i, j in zip(r, c):
                iou = 1.0 - cost[i, j]
                if iou >= self.iou_th:
                    self.tracks[i].box = dets_xyxy[j]
                    self.tracks[i].age = 0
                    matched_t.add(i); matched_d.add(j)

            for j, d in enumerate(dets_xyxy):
                if j not in matched_d:
                    self.tracks.append(Track(d, self.next_id))
                    self.next_id += 1

        else:
            for d in dets_xyxy:
                self.tracks.append(Track(d, self.next_id))
                self.next_id += 1

        self.tracks = [t for t in self.tracks if t.age <= self.max_age]
        return [(t.id, *t.box.tolist()) for t in self.tracks if t.age == 0]

In [ ]:
# Run detector on each MOT-val sequence frame and write MOTChallenge-style predictions.
from tqdm import tqdm

PRED_DIR = WORK / 'mot_predictions'
PRED_DIR.mkdir(exist_ok=True, parents=True)

model.eval()
SEQ_DIR = MOT_VAL / 'sequences'

for seq_path in sorted(SEQ_DIR.iterdir()):
    if not seq_path.is_dir():
        continue

    tracker = SORTLite(iou_th=0.3, max_age=10)
    rows = []
    frames = sorted((seq_path / 'img1').glob('*.jpg'))

    for fidx, img_path in enumerate(tqdm(frames, desc=seq_path.name), start=1):
        img = cv2.imread(str(img_path))[:, :, ::-1]
        tensor = torch.from_numpy(img).float().permute(2,0,1)[None] / 255.0
        tensor = tensor.to(DEVICE)

        with torch.no_grad():
            out = model(tensor)[0]

        boxes = out['boxes'].detach().cpu().numpy()
        scores = out['scores'].detach().cpu().numpy()

        keep = scores >= 0.4
        dets = boxes[keep]
        tracks = tracker.update(dets)

        for tid, x1, y1, x2, y2 in tracks:
            w, h = x2-x1, y2-y1
            rows.append(f'{fidx},{tid},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},1,-1,-1,-1')

    (PRED_DIR / f'{seq_path.name}.txt').write_text('\n'.join(rows))

print('Saved tracking predictions to', PRED_DIR)

## 7) Tracking Evaluation
Use TrackEval with VisDrone MOT-val GT and predictions to obtain **HOTA, MOTA, IDSW, Precision, Recall**.

In [ ]:
# Adjust GT folder names depending on your extracted VisDrone-MOT-val structure.
!python -m trackeval.run_mot_challenge \
  --BENCHMARK MOT17 \
  --SPLIT_TO_EVAL train \
  --GT_FOLDER $MOT_VAL/annotations \
  --SEQMAP_FILE $MOT_VAL/seqmaps/val.txt \
  --TRACKERS_FOLDER $WORK \
  --TRACKERS_TO_EVAL mot_predictions \
  --METRICS HOTA CLEAR Identity

## 8) Analysis Template (fill with your numbers)
- **What worked:**
  - Example: detector fine-tuning significantly improved recall over COCO-pretrained baseline.
- **Failure cases:**
  - Small/distant objects, fast camera motion, heavy occlusion, missed detections.
- **Tracking limitations:**
  - Motion-only association causes ID switches in crowded scenes.
- **What I would improve next:**
  - Better detector backbone / higher input resolution.
  - ByteTrack/DeepSORT with appearance embeddings.
  - Camera motion compensation and class-aware tracking thresholds.

## 9) Final Submission Checklist
- [ ] Notebook runs top-to-bottom in Colab.
- [ ] Detection metrics printed: Precision, Recall, mAP@0.50, mAP@0.50:0.95, mAR@0.50, mAR@0.50:0.95.
- [ ] Tracking metrics printed: HOTA, MOTA, IDSW, Precision, Recall.
- [ ] Brief design/limitations analysis included.
- [ ] Colab link permissions set to 'Anyone with the link can view'.